# LA Demographics
> This notebook fetches geographic data with related demographics from a Los Angeles County [Esri REST API endpoint](https://services.arcgis.com/RmCCgQtiZLDCtblq/ArcGIS/rest/services/Census_2020_SRR/FeatureServer) using the `ezesri` package. 

In [ ]:
gdf.

#### Load Python tools and Jupyter config

In [ ]:
import ezesri
import jenkspy
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from cartopy import crs as ccrs
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.font_manager as fm

# Set Roboto as the default font
plt.rcParams["font.family"] = "Roboto"

# Reference layers
la_city_boundary_url = "https://stilesdata.com/gis/la_city_boundary.geojson"
la_hoods_url = "https://stilesdata.com/gis/la_city_hoods_county_munis.geojson"

# Read neighborhoods and cities and exclude Catalina Island
la_hoods_src = gpd.read_file(la_hoods_url)
la_hoods_gdf = la_hoods_src.query('name.str.contains("Catalina|Avalon")')

# Define Esri service paths
blocks_url = "https://services.arcgis.com/RmCCgQtiZLDCtblq/ArcGIS/rest/services/Census_2020_SRR/FeatureServer/5"
tracks_url = (
    "https://maps.lacity.org/arcgis/rest/services/Mapping/NavigateLA/MapServer/59"
)

# Extract metadata and layer from service
metadata = ezesri.get_metadata(blocks_url)
gdf = ezesri.extract_layer(blocks_url).dropna(axis=0)

# Exclude Catalina and San Clemente islands
gdf = gdf_src.query("~ct20.isin(['599100', '599000'])")

# Lower case columns
gdf.columns = gdf.columns.str.lower()

# Which fields are available?
# for field in metadata["fields"]:
#     print(f'{field["name"].lower()}: {field["alias"]}')

# Mapping selected variables
columns_to_map = [
    "pc_nh_wht",
    "pc_nh_blk",
    "pc_nh_asn",
    "pc_hispanic",
    "pc_lessthan_hs",
    "pc_eng_below",
    "med_hh_incm",
    "pov100_20",
]

for column in columns_to_map:
    n_classes = 5
    breaks = jenkspy.jenks_breaks(gdf[f"{column}"], n_classes=n_classes)

    # Adjusted to align labels correctly
    teal_ramp = ["#f2f0f7", "#cbc9e2", "#9e9ac8", "#756bb1", "#54278f"]

    # Create custom colormaps
    teal_cmap = ListedColormap(teal_ramp)

    # Initialize plot
    fig, ax = plt.subplots(figsize=(10, 10))

    # Customize axes
    ax.set_title(f"{column}", fontsize=15)
    ax.axis("off")

    gdf.plot(
        ax=ax,
        column=column,
        cmap=teal_cmap,
        linewidth=0.1,
        edgecolor="gray",
        scheme="User_Defined",
        classification_kwds=dict(bins=breaks),
        legend=False,
    )

    # Plot state boundaries
    la_hoods_gdf.boundary.plot(ax=ax, linewidth=0.3, color="white")

    # Adjust colorbars to be narrower, centered relative to the map and reduce space
    key_width = 0.25  # Narrower width to avoid bumping
    key_bottom = -0.05  # Move the colorbars closer to the map

    key_ax = fig.add_axes([0.55, key_bottom, key_width, 0.02])  # Key position

    tickBreaks = breaks
    tickLabels = breaks

    # Define key
    key = fig.colorbar(
        plt.cm.ScalarMappable(cmap=teal_cmap, norm=BoundaryNorm(breaks, teal_cmap.N)),
        cax=key_ax,
        orientation="horizontal",
        ticks=tickBreaks,
    )
    key.set_label(f"{column}", fontsize=10)
    key.ax.set_xticklabels(tickLabels, fontsize=8)

    # Save the plot
    plt.savefig(f"visuals/lacounty_demographics_map_{column}.png", bbox_inches="tight")
    plt.close()

    gdf.to_file("data/processed/lacounty_demographics_blocks.geojson", driver="GeoJSON")